In [37]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
from prophet import Prophet

import warnings
import prophet
from prophet.diagnostics import cross_validation, performance_metrics
import matplotlib as plt

import pandas as pd
import numpy as np
import seaborn as sns
from loguru import logger
import sys

warnings.filterwarnings("ignore", category=FutureWarning)
logger.remove()
logger.add(sys.stderr, level="INFO", format="{time:HH:mm:ss} | {level:<7} | {message}")
logger.info("FE Avance 2 — EpiForecast-MX inicializado")

12:50:04 | INFO    | FE Avance 2 — EpiForecast-MX inicializado


In [2]:
# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# --- Rutas -------------------------------------------------------------------
# Dataset ya preparado para hacer merge con INEGI (sale del pipeline de limpieza/transform)
DATA_PATH = Path("../data/processed/data_inegi_General.csv")

# --- Paleta IMSS institucional (Guía cromática oficial) ----------------------
IMSS_COLORS = {
    "neutral_black":  "#231F20",  # PANTONE Neutral Black C
    "burgundy":       "#9B2242",  # PANTONE 7420 C
    "dark_burgundy":  "#6F1D46",  # PANTONE 7421 C
    "cool_gray":      "#97999B",  # PANTONE Cool Gray C
    "teal":           "#00524E",  # PANTONE IMSS 561 C
    "dark_teal":      "#173F35",  # PANTONE 627 C
    "cream":          "#E8D5B5",  # PANTONE 7402 C
    "gold":           "#B58500",  # PANTONE 1255 C
}

# Paleta secuencial para gráficos
PALETTE_MAIN = [
    IMSS_COLORS["teal"],
    IMSS_COLORS["burgundy"],
    IMSS_COLORS["gold"],
    IMSS_COLORS["dark_teal"],
    IMSS_COLORS["dark_burgundy"],
    IMSS_COLORS["cool_gray"],
    IMSS_COLORS["neutral_black"],
    IMSS_COLORS["cream"],
]

PALETTE_PADECIMIENTO = {
    "Depresión":  IMSS_COLORS["burgundy"],
    "Parkinson":  IMSS_COLORS["teal"],
    "Alzheimer":  IMSS_COLORS["gold"],
}

PALETTE_SEXO = {
    "Hombres": IMSS_COLORS["teal"],
    "Mujeres": IMSS_COLORS["burgundy"],
}

# --- Estilo global de matplotlib ---------------------------------------------
plt.rcParams.update({
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.edgecolor":      IMSS_COLORS["cool_gray"],
    "axes.labelcolor":     IMSS_COLORS["neutral_black"],
    "text.color":          IMSS_COLORS["neutral_black"],
    "xtick.color":         IMSS_COLORS["neutral_black"],
    "ytick.color":         IMSS_COLORS["neutral_black"],
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "grid.color":          IMSS_COLORS["cool_gray"],
    "font.family":         "sans-serif",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "figure.titlesize":    15,
    "figure.titleweight":  "bold",
    "figure.dpi":          120,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
})

logger.success(f"Configuración cargada | Paleta IMSS: {len(IMSS_COLORS)} colores")

12:30:25 | SUCCESS | Configuración cargada | Paleta IMSS: 8 colores


In [3]:
df_datos = pd.read_csv(DATA_PATH)
df_datos['Fecha'] = pd.to_datetime(df_datos['Fecha'])
df_datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60384 entries, 0 to 60383
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Padecimiento                        60384 non-null  object        
 1   Semana                              60384 non-null  int64         
 2   Fecha                               60384 non-null  datetime64[ns]
 3   Entidad                             60384 non-null  object        
 4   incrementos_hombres                 60384 non-null  int64         
 5   incrementos_mujeres                 60384 non-null  int64         
 6   Region                              60384 non-null  object        
 7   Superficie_km2                      60384 non-null  float64       
 8   Hombres                             60384 non-null  int64         
 9   Mujeres                             60384 non-null  int64         
 10  Total                 

In [56]:
serie  = (df_datos
                .groupby(["Fecha","Entidad"])[["incrementos_hombres", "incrementos_mujeres"]]
                .sum()
                .reset_index()
                .rename(columns={'Fecha':'ds'})
            )

serie["y"] = serie["incrementos_hombres"] + serie["incrementos_mujeres"]
serie = serie.sort_values('ds')
regiones = serie['Entidad'].unique()
serie.head(5)

,ds,Entidad,incrementos_hombres,incrementos_mujeres,y
0,2013-12-30,Aguascalientes,1,0,1
31,2013-12-30,Zacatecas,0,1,1
30,2013-12-30,Yucatán,8,17,25
29,2013-12-30,Veracruz,8,9,17
28,2013-12-30,Tlaxcala,0,0,0


In [59]:

resultados = []

for region in regiones:
    modelo = Prophet()
    serie_prophet = serie.loc[serie['Entidad']==region,['ds','y']].copy()
    modelo.fit(serie_prophet)

    cv_prophet = cross_validation(
        modelo,
        initial='360 days',    
        period='30 days',
        horizon='90 days'
    )

    df_pm = performance_metrics(cv_prophet)
    
    den = cv_prophet['y'].replace(0, np.nan)  # evitar división entre 0
    mape_series = np.abs((cv_prophet['y'] - cv_prophet['yhat']) / den) * 100
    mape_mean = mape_series.mean()  # promedio ignorando NaN

    metricas_disponibles = [c for c in ['rmse', 'mae', 'mdape'] if c in df_pm.columns]
    resumen_base = df_pm[metricas_disponibles].mean(numeric_only=True)

    resumen_completo = resumen_base.to_dict()
    resumen_completo['mape'] = mape_mean

    orden = ['rmse', 'mae', 'mape', 'mdape']
    resumen_ordenado = [resumen_completo.get(k, np.nan) for k in orden]

    fila = [region] + resumen_ordenado
    resultados.append(fila)



13:21:27 - cmdstanpy - INFO - Chain [1] start processing
13:21:27 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
  0%|          | 0/132 [00:00<?, ?it/s]13:21:27 - cmdstanpy - INFO - Chain [1] start processing
13:21:27 - cmdstanpy - INFO - Chain [1] done processing
  1%|          | 1/132 [00:00<00:37,  3.45it/s]13:21:27 - cmdstanpy - INFO - Chain [1] start processing
13:21:28 - cmdstanpy - INFO - Chain [1] done processing
  2%|▏         | 2/132 [00:00<00:36,  3.55it/s]13:21:28 - cmdstanpy - INFO - Chain [1] start processing
13:21:28 - cmdstanpy - INFO - Chain [1] done processing
  2%|▏         | 3/132 [00:00<00:32,  4.01it/s]13:21:28 - cmdstanpy - INFO - Chain [1] start processing
13:21:28 - cmdstanpy - INFO - Chain [1] done processing
  3%|▎         | 4/132 [00:01<00:32,  3.92it/s]13:21:28 - cmdstanpy - INFO - Chain [1] start processing
13:21:28 - cmdstanpy - INFO - Chain [1] done pr

In [60]:
df_resultados = pd.DataFrame(resultados, columns=['region', 'rmse', 'mae', 'mape', 'mdape'])
print(df_resultados)


                 region       rmse        mae       mape     mdape
0        Aguascalientes  17.140651  12.532016  54.123146  0.307596
1             Zacatecas  13.836217   9.796258  32.787017  0.225804
2               Yucatán  21.131182  15.745711  52.431058  0.286587
3              Veracruz  37.058729  28.548598  32.303847  0.179272
4              Tlaxcala  10.481668   6.913262  47.812673  0.292000
5            Tamaulipas  27.994514  21.902449  31.235929  0.211515
6               Tabasco  29.661524  21.551543  57.845665  0.333993
7                Sonora  23.033181  16.704411  37.118180  0.230508
8               Sinaloa  25.815358  19.986598  25.680400  0.181209
9          Quintana Roo  14.123728   9.825662  57.450190  0.280789
10            Querétaro  13.467546   8.499362  51.401713  0.321346
11               Puebla  33.679113  24.646358  67.680115  0.289644
12               Oaxaca  19.391720  14.021995  50.573718  0.313595
13           Nuevo León  28.098878  21.854356  31.064740  0.19